# 🚧 CivicMind — Training Notebook
### Multi-Agent Pothole Repair Scheduler

This notebook trains an LLM agent on the CivicMind environment hosted on HuggingFace Spaces.

**Runtime:** T4 GPU (Runtime → Change runtime type → T4 GPU)

In [ ]:
# Install dependencies
!pip install -q trl>=0.9.0 transformers>=4.40.0 requests matplotlib accelerate peft
print("✅ Dependencies installed")

## ⚙️ Config

In [ ]:
import os, json, random, time, requests
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from typing import List, Dict, Tuple, Optional
import torch

CONFIG = {
    "hf_space_url": "https://meet25284-pothole-repair-env.hf.space",
    "model_name": "Qwen/Qwen2.5-0.5B-Instruct",
    "episodes": 200,
    "max_steps_per_episode": 30,
    "tasks": ["critical_repair", "budget_optimizer", "full_city_manager"],
    "success_threshold": 0.5,
    "learning_rate": 5e-6,
    "batch_size": 2,
    "max_new_tokens": 30,
    "temperature": 0.7,
    "seed": 42,
}

random.seed(CONFIG["seed"])
print("✅ Config loaded")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## 🌐 Environment Client
This class talks to your HuggingFace Space via HTTP.

In [ ]:
class PotholeEnvClient:
    def __init__(self, base_url: str):
        self.base_url = base_url.rstrip("/")
        self.session = requests.Session()

    def _post(self, endpoint: str, data: dict, retries: int = 3) -> dict:
        for attempt in range(retries):
            try:
                r = self.session.post(f"{self.base_url}{endpoint}", json=data, timeout=30)
                r.raise_for_status()
                return r.json()
            except Exception as e:
                if attempt == retries - 1:
                    print(f"[ERROR] {endpoint} failed: {e}")
                    return {}
                time.sleep(2)
        return {}

    def _get(self, endpoint: str, retries: int = 3) -> dict:
        for attempt in range(retries):
            try:
                r = self.session.get(f"{self.base_url}{endpoint}", timeout=30)
                r.raise_for_status()
                return r.json()
            except Exception as e:
                if attempt == retries - 1:
                    print(f"[ERROR] {endpoint} failed: {e}")
                    return {}
                time.sleep(2)
        return {}

    def health(self) -> bool:
        result = self._get("/health")
        return result.get("status") == "ok"

    def reset(self, task_name: str = "critical_repair") -> dict:
        return self._post("/reset", {"task_name": task_name})

    def step(self, action_type: str, pothole_id: str) -> dict:
        return self._post("/step", {
            "action_type": action_type,
            "pothole_id": pothole_id,
            "defer_days": 1
        })

    def get_score(self) -> float:
        result = self._get("/score")
        return float(result.get("score", 0.0))

# Connect and test
env_client = PotholeEnvClient(CONFIG["hf_space_url"])
if env_client.health():
    print("✅ Connected to HF Space!")
    obs = env_client.reset("critical_repair")
    print(f"✅ Reset works — {len(obs.get('potholes', []))} potholes loaded")
else:
    print("❌ Cannot connect — check your HF Space URL")

## 📝 Prompt Builder & Action Parser

In [ ]:
def build_prompt(obs: dict, task_name: str) -> str:
    potholes = obs.get("potholes", [])
    pending = [p for p in potholes if p.get("status") == "pending"]
    pending_sorted = sorted(pending, key=lambda p: (-p.get("severity", 0), -p.get("daily_traffic", 0)))[:5]

    weather = obs.get("weather", {})
    rain_warn = "RAINING — avoid dispatch!" if weather.get("is_raining") else "Clear"

    lines = []
    for p in pending_sorted:
        sev = p.get("severity", 0)
        sev_str = "UNKNOWN" if sev == 0 else str(sev)
        lines.append(f"  {p.get('id')} | sev={sev_str} | {p.get('road_type')} | traffic={p.get('daily_traffic',0):,} | cost=Rs{p.get('repair_cost',0):.0f}")

    pothole_block = "\n".join(lines) if lines else "  None pending"

    return f"""You are a city road repair agent.
Task: {task_name}
Day: {obs.get('day',1)} of {obs.get('max_days',30)}
Budget: Rs{obs.get('budget_remaining',0):,.0f}
Crews: {obs.get('crews_available',0)}
Weather: {rain_warn}
Fixed: {obs.get('total_fixed',0)} | Pending: {obs.get('total_pending',0)}

Top pending potholes:
{pothole_block}

Choose ONE action. Reply with EXACTLY:
dispatch POT_001
or defer POT_001
or mark_low POT_001

Your action:"""

def parse_action(response: str, obs: dict) -> Tuple[str, str]:
    potholes = obs.get("potholes", [])
    pending_ids = [p.get("id") for p in potholes if p.get("status") == "pending"]
    fallback_id = pending_ids[0] if pending_ids else "POT_001"
    try:
        text = response.strip().lower().split("\n")[0].strip()
        parts = text.split()
        if len(parts) < 2:
            return "defer", fallback_id
        verb = parts[0].strip()
        pot_id = parts[1].strip().upper()
        valid = {"dispatch": "dispatch", "defer": "defer", "mark_low": "mark_low_priority", "mark_low_priority": "mark_low_priority"}
        action_type = valid.get(verb)
        if not action_type:
            return "defer", fallback_id
        all_ids = [p.get("id") for p in potholes]
        if pot_id not in all_ids:
            pot_id = fallback_id
        return action_type, pot_id
    except Exception:
        return "defer", fallback_id

# Test
test_obs = env_client.reset("critical_repair")
print("=== Sample Prompt ===")
print(build_prompt(test_obs, "critical_repair"))

## 🏆 Reward Shaper

In [ ]:
def shape_reward(step_result: dict, action_type: str, pothole_id: str, obs: dict) -> float:
    base_reward = float(step_result.get("reward", 0.0))
    bonus = 0.0
    potholes = obs.get("potholes", [])
    pothole = next((p for p in potholes if p.get("id") == pothole_id), None)
    weather = obs.get("weather", {})
    is_raining = weather.get("is_raining", False)

    if pothole:
        severity = pothole.get("severity", 0)
        road_type = pothole.get("road_type", "")
        if action_type == "dispatch":
            if severity >= 4: bonus += 0.15
            if road_type == "highway": bonus += 0.10
            if is_raining: bonus -= 0.25
        elif action_type == "defer":
            if severity >= 4: bonus -= 0.15
            if is_raining and severity < 4: bonus += 0.05
        elif action_type == "mark_low_priority":
            if severity >= 4: bonus -= 0.30
            elif severity <= 2: bonus += 0.05

    return float(max(min(base_reward + bonus, 1.0), -1.0))

print("✅ Reward shaper ready")

## 🤖 Load Model

> This takes 2-3 minutes on first run.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading: {CONFIG['model_name']}")

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"], trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

device = next(model.parameters()).device
print(f"✅ Model loaded on {device}")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 📊 Random Baseline
Run a random agent first so we have something to compare against.

In [ ]:
def random_baseline_episode(task_name: str) -> float:
    obs = env_client.reset(task_name)
    for _ in range(CONFIG["max_steps_per_episode"]):
        potholes = obs.get("potholes", [])
        pending = [p for p in potholes if p.get("status") == "pending"]
        if not pending:
            break
        action_type = random.choice(["dispatch", "defer", "mark_low_priority"])
        pothole_id = random.choice(pending).get("id")
        result = env_client.step(action_type, pothole_id)
        obs = result.get("observation", obs)
        if result.get("done", False):
            break
    return env_client.get_score()

print("Collecting baseline scores...")
baseline_scores = []
for task in CONFIG["tasks"]:
    score = random_baseline_episode(task)
    baseline_scores.append(score)
    print(f"  {task}: {score:.3f}")

baseline_avg = sum(baseline_scores) / len(baseline_scores)
print(f"\nBaseline average: {baseline_avg:.3f}")

## 🚀 Training Loop

This trains the model using REINFORCE policy gradient. Each episode the model interacts with your live HF Space environment.

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=CONFIG["learning_rate"])
episode_scores = []
episode_tasks  = []

model.train()
print(f"Starting training for {CONFIG['episodes']} episodes...")
print("-" * 60)

for episode in range(CONFIG["episodes"]):
    task_name = CONFIG["tasks"][episode % 3]
    obs = env_client.reset(task_name)

    rewards_list = []

    for step in range(CONFIG["max_steps_per_episode"]):
        potholes = obs.get("potholes", [])
        pending = [p for p in potholes if p.get("status") == "pending"]
        if not pending:
            break

        prompt = build_prompt(obs, task_name)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=CONFIG["max_new_tokens"],
                temperature=CONFIG["temperature"],
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        action_type, pothole_id = parse_action(response, obs)

        result = env_client.step(action_type, pothole_id)
        reward = shape_reward(result, action_type, pothole_id, obs)
        rewards_list.append(reward)
        obs = result.get("observation", obs)

        if result.get("done", False):
            break

    final_score = env_client.get_score()
    episode_scores.append(final_score)
    episode_tasks.append(task_name)

    # REINFORCE update
    if rewards_list:
        returns = []
        G = 0.0
        for r in reversed(rewards_list):
            G = r + 0.99 * G
            returns.insert(0, G)
        returns_t = torch.tensor(returns, dtype=torch.float32, requires_grad=True)
        if len(returns_t) > 1:
            returns_t = (returns_t - returns_t.mean()) / (returns_t.std() + 1e-8)
        loss = -returns_t.mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (episode + 1) % 10 == 0:
        avg10 = sum(episode_scores[-10:]) / min(10, len(episode_scores))
        print(f"Episode {episode+1:3d}/{CONFIG['episodes']} | task={task_name:22s} | score={final_score:.3f} | avg10={avg10:.3f}")

trained_avg = sum(episode_scores[-20:]) / min(20, len(episode_scores))
print(f"\n✅ Training complete!")
print(f"   Baseline avg : {baseline_avg:.3f}")
print(f"   Trained avg  : {trained_avg:.3f}")
print(f"   Improvement  : +{((trained_avg - baseline_avg) / max(baseline_avg, 0.001)) * 100:.1f}%")

## 📈 Save Plots

These plots must be committed to your repo and embedded in README.

In [ ]:
import os
os.makedirs("plots", exist_ok=True)

def smooth(values, window=10):
    result = []
    for i in range(len(values)):
        start = max(0, i - window + 1)
        result.append(sum(values[start:i+1]) / (i - start + 1))
    return result

# ── Plot 1: Reward Curve ──
fig, ax = plt.subplots(figsize=(10, 5))
xs = list(range(1, len(episode_scores) + 1))
smoothed = smooth(episode_scores, 10)

ax.axhline(y=baseline_avg, color="gray", linestyle="--", linewidth=1.5, label=f"Random baseline ({baseline_avg:.2f})")
ax.plot(xs, episode_scores, color="#90EE90", alpha=0.3, linewidth=0.8, label="Per-episode score")
ax.plot(xs, smoothed, color="#1D9E75", linewidth=2.5, label="Smoothed (window=10)")
ax.axhline(y=0.5, color="#534AB7", linestyle=":", linewidth=1.5, label="Success threshold (0.5)")

ax.set_xlabel("Training Episode", fontsize=12)
ax.set_ylabel("Episode Score (0.0 – 1.0)", fontsize=12)
ax.set_title("CivicMind — Agent Learning Progress", fontsize=13)
ax.legend(loc="lower right", fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.1, 1.1)
plt.tight_layout()
plt.savefig("plots/reward_curve.png", dpi=150, bbox_inches="tight")
print("✅ Saved plots/reward_curve.png")
plt.show()

# ── Plot 2: Before vs After ──
task_labels = ["Easy\n(critical)", "Medium\n(budget)", "Hard\n(city)"]
task_colors = ["#1D9E75", "#BA7517", "#D85A30"]

trained_per_task = []
for task in CONFIG["tasks"]:
    task_sc = [s for s, t in zip(episode_scores[-60:], episode_tasks[-60:]) if t == task]
    trained_per_task.append(sum(task_sc) / max(len(task_sc), 1))

fig, ax = plt.subplots(figsize=(8, 5))
x = range(3)
width = 0.35
b1 = ax.bar([i - width/2 for i in x], [baseline_avg]*3, width, label="Random baseline", color="gray", alpha=0.7)
b2 = ax.bar([i + width/2 for i in x], trained_per_task, width, label="Trained agent", color=task_colors, alpha=0.9)

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{bar.get_height():.2f}", ha="center", fontsize=9)

ax.set_xlabel("Task", fontsize=12)
ax.set_ylabel("Average Score", fontsize=12)
ax.set_title("CivicMind — Before vs After Training", fontsize=13)
ax.set_xticks(list(x))
ax.set_xticklabels(task_labels)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("plots/before_after.png", dpi=150, bbox_inches="tight")
print("✅ Saved plots/before_after.png")
plt.show()

print("\n=== Download plots from left sidebar → plots/ folder ===")
print("Then commit them to your GitHub repo!")

## 💾 Save & Upload Model

In [ ]:
# Save model locally
save_dir = "./civicmind_trained_model"
model.eval()
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"✅ Model saved to {save_dir}")

# Upload to HuggingFace Hub
HF_USERNAME = "meet25284"  # change this
HF_TOKEN    = ""           # paste your HF token here

if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi()
    api.upload_folder(
        folder_path=save_dir,
        repo_id=f"{HF_USERNAME}/civicmind-agent",
        repo_type="model",
        token=HF_TOKEN,
    )
    print(f"✅ Model uploaded to huggingface.co/{HF_USERNAME}/civicmind-agent")
else:
    print("Add your HF_TOKEN above to upload the model")